# Demo 17 — Per-group dispersion (`dispersion` / `dispformula`)

A Gamma GLM assumes a constant coefficient of variation: the scatter scales with
the mean through a single dispersion parameter. When groups differ not only in
level but in *relative* scatter, that one dispersion is a poor compromise, and the
standard errors it produces are mis-scaled.

`options.dispersion` sets the right-hand side of glmmTMB's `dispformula`, so the
dispersion may vary by a factor instead of the default constant `~ 1`. Here, on the
`ToothGrowth` dataset: odontoblast length of 60 guinea pigs given vitamin C as
orange juice (`OJ`) or ascorbic acid (`VC`) at three ordered doses (low, medium,
high). The doses differ markedly in relative scatter (coefficient of variation
about 0.42 at low dose but only about 0.15 at high), so one dispersion cannot fit
all three well.

We fit `len ~ supp * dose` twice — constant dispersion, then dispersion varying by
dose — and compare the fit.

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## The model

The crossed two-way Gamma model `len ~ supp * dose`, shared by both fits. `out_dir`
is empty, so results render inline only.

In [ ]:
def base_options():
    o = KbstatOptions()
    o.in_file      = os.path.join(o.demo_dir, 'data/toothgrowth.csv')
    o.out_dir      = ''            # inline only; set a folder to also save
    o.y            = 'len'
    o.y_units      = 'mm'
    o.x            = 'supp, dose'
    o.interaction  = 'supp, dose'
    o.x_order      = 'dose: low, medium, high'
    o.distribution = 'gamma'
    o.link         = 'log'
    return o

## 1. Constant dispersion (default)

With the default `dispformula = ~ 1`, one dispersion parameter is shared across all
doses.

In [ ]:
shared = base_options()
kb_shared = Kbstat(shared)
kb_shared.run();

## 2. Dispersion varying by dose

`dispersion = 'dose'` maps to glmmTMB `dispformula = ~ dose`, so each dose gets its
own dispersion. The mean model is unchanged.

In [ ]:
by_dose = base_options()
by_dose.dispersion = 'dose'
kb_by_dose = Kbstat(by_dose)
kb_by_dose.run();

## Compare the fit

Letting the dispersion vary by dose lowers the AIC substantially here (a clearly
better fit), while the mean structure and the estimated marginal means are
unchanged.

In [ ]:
aic_shared  = float(kb_shared.model.fit_stats['AIC'].iloc[0])
aic_by_dose = float(kb_by_dose.model.fit_stats['AIC'].iloc[0])
print(f'AIC  shared dispersion (~ 1)     : {aic_shared:.1f}')
print(f'AIC  dispersion by dose (~ dose) : {aic_by_dose:.1f}')
print(f'delta AIC (negative favours ~ dose): {aic_by_dose - aic_shared:+.1f}')

## Interpretation

- Both models return the **same** estimated marginal means and the same group
  directions; only the **dispersion**, and therefore the standard errors,
  confidence intervals and p-values, differ.
- The lower AIC for `~ dose` says the constant-dispersion assumption was a poor
  fit: the low-dose group is far more variable (relative to its mean) than the
  high-dose group.
- Use `dispersion` whenever pooled groups differ widely in scale or scatter (for
  example joint regions with very different torque magnitudes). It is ignored for
  gaussian (LM/LMM) models, which estimate their own residual variance.